In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
import tensorly as tl
import cell2cell as c2c
import liana as li

# Suppress warnings for clean output
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# DIRECTORY SETUP
# ---------------------------------------------------------
# Use relative paths for GitHub reproducibility
LIANA_DIR = './results/tables/liana/'
OUTPUT_TENSOR = './results/tensor_outputs/'
os.makedirs(OUTPUT_TENSOR, exist_ok=True)

# ---------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------
def get_context_metadata():
    """
    Define the mapping of sample IDs to their respective disease contexts (HCC vs iCCA).
    """
    from collections import defaultdict
    context_dict = defaultdict(lambda: 'Unknown')

    # Pre-defined clinical mapping
    mapping = {
        'H08': 'HCC', 'H70': 'HCC', 'H72': 'HCC', 'H73a': 'HCC', 'H74': 'HCC',
        'H73b': 'HCC', 'H75': 'HCC', 'H49b': 'HCC', 'H77': 'HCC', 'H68a': 'HCC',
        'H58c': 'HCC', 'H68b': 'HCC', 'H55': 'HCC', 'H62': 'HCC', 'H34b': 'HCC',
        'H34a': 'HCC', 'H34c': 'HCC', 'H41': 'HCC', 'H43': 'HCC', 'H49a': 'HCC',
        'H58a': 'HCC', 'H58b': 'HCC', 'H63': 'HCC', 'H65': 'HCC', 'H21': 'HCC',
        'H28': 'HCC', 'H23': 'HCC', 'H30': 'HCC', 'H38': 'HCC', 'H18': 'HCC',
        'H37': 'HCC',
        'C76': 'ICCa', 'C60': 'ICCa', 'C56': 'ICCa', 'C26a': 'ICCa', 'C42': 'ICCa',
        'C46b': 'ICCa', 'C66': 'ICCa', 'C26b': 'ICCa', 'C25': 'ICCa', 'C29': 'ICCa',
        'C39': 'ICCa', 'C35': 'ICCa'
    }
    context_dict.update(mapping)
    return context_dict

def construct_4d_tensor(liana_file_path, output_dir):
    """
    Construct a 4D communication tensor (Contexts x LR pairs x Senders x Receivers)
    from sample-specific LIANA results.
    """
    print(f"[*] Loading LIANA results from {liana_file_path}...")
    liana_res = pd.read_csv(liana_file_path)
    sorted_samples = sorted(liana_res['Sample'].unique())

    print("[*] Building 4D Tensor using cell2cell...")
    tensor = li.multi.to_tensor_c2c(
        liana_res=liana_res,
        sample_key='Sample',
        source_key='source',
        target_key='target',
        ligand_key='ligand_complex', # Updated to match LIANA default if necessary
        receptor_key='receptor_complex', # Updated to match LIANA default if necessary
        score_key='magnitude_rank',
        non_negative=True,
        inverse_fun=lambda x: 1 - x, # Transform rank to weight
        non_expressed_fill=None,
        how='outer',
        lr_fill=np.nan,
        cell_fill=np.nan,
        outer_fraction=1/3.,
        lr_sep='^',
        context_order=sorted_samples,
        sort_elements=True
    )

    print("[*] Generating Tensor Metadata...")
    context_dict = get_context_metadata()
    meta_tensor = c2c.tensor.generate_tensor_metadata(
        interaction_tensor=tensor,
        metadata_dicts=[context_dict, None, None, None],
        fill_with_order_elements=True
    )

    # Export raw tensors for reproducibility
    tensor_path = os.path.join(output_dir, 'cancer-Tensor.pkl')
    meta_path = os.path.join(output_dir, 'cancer-Tensor-Metadata.pkl')

    c2c.io.export_variable_with_pickle(tensor, tensor_path)
    c2c.io.export_variable_with_pickle(meta_tensor, meta_path)
    print(f"[*] Tensor successfully exported to {output_dir}")

    return tensor, meta_tensor


def run_tensor_decomposition(tensor, meta_tensor, output_dir):
    """
    Perform Non-negative Tensor Factorization (NTF) using Tensor-cell2cell.
    Automatically identifies the optimal rank using Elbow analysis.
    """
    # Auto-detect hardware accelerator (GPU/CPU)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"[*] Initializing PyTorch backend on device: {device.upper()}")

    if device == 'cuda':
        tl.set_backend('pytorch')

    print("[*] Running Tensor-cell2cell Pipeline (This may take a while)...")
    tensor_res = c2c.analysis.run_tensor_cell2cell_pipeline(
        tensor,
        meta_tensor,
        copy_tensor=True,
        rank=None, # Auto-determine via elbow analysis
        tf_optimization='robust',
        random_state=0,
        device=device,
        elbow_metric='error',
        smooth_elbow=False,
        upper_rank=25,
        tf_init='random',
        tf_svd='numpy_svd',
        cmaps=None,
        sample_col='Element',
        group_col='Category',
        fig_fontsize=14,
        output_folder=output_dir,
        output_fig=True,
        fig_format='pdf'
    )

    print("[*] Tensor Factorization complete! Elbow plot and loadings saved.")
    return tensor_res

# =========================================================
# EXECUTION PIPELINE
# =========================================================
LIANA_FILE = os.path.join(LIANA_DIR, "LIANA_by_sample.csv")

# 1. Build Tensor
tensor, meta_tensor = construct_4d_tensor(liana_file_path=LIANA_FILE, output_dir=OUTPUT_TENSOR)

# 2. Decompose Tensor
# Note: Results (Loadings.xlsx and Elbow.pdf) will be automatically saved to OUTPUT_TENSOR
tensor_res = run_tensor_decomposition(tensor, meta_tensor, output_dir=OUTPUT_TENSOR)